# 第 25 章：动量（Momentum）

本 Notebook 复现正文第 8–12 节：12–1 的精确窗口与三种错误窗口、求和对复利的排序反转、横截面与时间序列组合的手算、重叠持有的换手公式、拆股与展期的算术，以及 French 10 行业上的真实结果（窗口变体、$J\times K$ 网格、时间序列对始终做多、崩溃状态）。

- 手算例是教学构造；French 行业与 FF3 是 2026-10-05 冻结的已修订历史快照（10 个行业组合，不是个股，不是成交）。
- 核心函数来自 `src/quant_course/factor_signals.py`、`momentum.py`、`factor_research.py`；章节 CLI 在 `code/ch25_momentum.py`。本 Notebook 不写文件，须从第一个单元顺序运行。

**I2 默认离线路线：CONSTRUCTED 教学计算。** French 数据未随包提供，权利/用途 unknown；真实市场分支显式跳过，不能据此声称完整实证复现。原文历史数字只是章节背景，非本次输出。A10-A design accepted / execution blocked；A10-A.1 protocol technically accepted pending permissions and execution authorization；A10-B 未授权。静态图如显示则为已冻结教学参考，不表示本次重新生成。


In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch25_momentum_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. 窗口端点与复利（8.1、6.2）

In [ ]:
import numpy as np
import pandas as pd
from quant_course.factor_signals import formation_return

r = [0.03, -0.02, 0.05, 0.01, -0.04, 0.06, 0.02, -0.01, 0.04, 0.03, -0.02, 0.05, -0.06, 0.02]
frame = pd.DataFrame({"a": r}, index=pd.period_range("2020-01", periods=14, freq="M"))
s = pd.Period("2021-01", "M")
rows = {"12-1 (months 2..12)": formation_return(frame, 11, 1).loc[s, "a"], "12-0 (months 2..13)": formation_return(frame, 12, 0).loc[s, "a"],
        "shift earlier (months 1..11)": np.prod(1 + np.array(r[0:11])) - 1, "shift later (months 3..13)": np.prod(1 + np.array(r[2:13])) - 1,
        "sum of months 2..12": sum(r[1:12])}
out = pd.Series(rows)
assert abs(out["12-1 (months 2..12)"] - 0.177159) < 1e-6
out.round(4)

In [ ]:
two = pd.DataFrame({"X": [0.5, -0.5], "Y": [0.0, 0.0]}, index=pd.period_range("2020-01", periods=2, freq="M"))
print("sums:", two.sum().to_dict(), "| compound:", formation_return(two, 2, 0).iloc[-1].round(4).to_dict())

## 2. 横截面六资产、时间序列波动率缩放（8.2、8.3）

In [ ]:
from quant_course.factor_research import quantile_long_short_weights
from quant_course.momentum import tsmom_weights

idx1 = pd.period_range("2026-06", periods=1, freq="M")
formation = pd.DataFrame([[0.30, 0.12, 0.05, -0.02, -0.10, -0.25]], index=idx1, columns=list("ABCDEF"))
w = quantile_long_short_weights(formation, 3).iloc[0]
nxt = np.array([0.02, 0.03, 0.01, 0.00, 0.01, -0.02])
sig = pd.DataFrame([[1.0, -1.0, 1.0]], index=idx1, columns=list("xyz")); vol = pd.DataFrame([[0.10, 0.20, 0.40]], index=idx1, columns=list("xyz"))
tw = tsmom_weights(sig, vol, target_vol=0.40).iloc[0]
print("CS long-short:", round(float(w.to_numpy() @ nxt), 4), "| TS weights:", tw.round(3).to_dict(), "gross", round(tw.abs().sum(), 3), "net", round(tw.sum(), 3))
assert abs(w.to_numpy() @ nxt - 0.03) < 1e-12 and abs(tw.sum() - 1.0) < 1e-12

## 3. 重叠持有：换手 = ||w_s − w_(s−K)||₁ / K（6.3）

In [ ]:
from quant_course.momentum import blend_vintages
rng = np.random.default_rng(2)
vint = pd.DataFrame(rng.choice([-0.5, 0.0, 0.5], size=(24, 6)), index=pd.period_range("2020-01", periods=24, freq="M"))
for K in (1, 3, 6):
    b = blend_vintages(vint, K)
    turn = b.diff().abs().sum(axis=1, min_count=1)
    formula = pd.Series([np.abs(vint.iloc[i] - vint.iloc[i - K]).sum() / K for i in range(len(vint))], index=vint.index)
    assert np.allclose(turn.iloc[K:], formula.iloc[K:], atol=1e-12)
print("formula verified for K = 1, 3, 6")

## 4. French 10 行业：窗口变体、网格、时间序列、崩溃状态

In [ ]:
if RUN_FRENCH:
    ch25 = load_code_module("ch25_momentum.py")
    real = ch25.industry_study()
    cs = pd.DataFrame({K: {"gross mean %": v["gross"]["mean_pct"], "gross t": v["gross"]["t_nw"], "net mean %": v["net"]["mean_pct"], "turnover": v["mean_turnover"]}
                       for K, v in real["cs_12_1"].items()}).T
    assert abs(real["cs_12_1"]["1"]["gross"]["t_nw"] - 3.64) < 0.005 and real["worst_10_in_bear"] == 7
    cs.round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    variants = pd.DataFrame(real["cs_window_variants"]).T[["months", "mean_pct", "t_nw"]]
    grid = pd.DataFrame({J: {K: real["grid_J_K"][f"J{J}_K{K}"]["t_nw"] for K in (1, 3, 6, 12)} for J in (3, 6, 9, 12)}).T
    print("Bonferroni z for 16 strategies:", round(real["grid_bonferroni_hurdle_z_16"], 3))
    variants.round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    grid.round(2)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    ts = pd.DataFrame(real["ts_momentum"]).T[["mean_pct", "t_nw", "sd_pct", "sharpe_annual", "mean_gross_exposure"]]
    print("TS 12-0 minus always long:", {k: round(v, 3) for k, v in real["ts_minus_always_long"].items() if k in ("mean_pct", "t_nw")}, "| corr(CS, TS) =", round(real["corr_cs_ts"], 3))
    assert real["ts_momentum"]["always long (same scaling)"]["mean_pct"] > real["ts_momentum"]["TS 12-0"]["mean_pct"]
    ts.round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    print(real["bear_state"])
    pd.DataFrame(real["worst_10_months"]).round(2)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 5. 自己试一试

1. 把第 4 节的成本 `ch25.COST_PER_UNIT` 改成 0.0020，重新运行 `industry_study()`，看 $K=1$ 与 $K=3$ 的净 $t$ 谁先降到 2 以下。
2. 在第 1 节把窗口改为 `formation_return(frame, 10, 2)`（跳过两个月），解释信号为何与 12–1 不同。
3. 把熊市状态的窗口从 24 个月改为 12 个月（修改 `industry_study` 中的 `window=24`），看最差月份里有多少个仍处于熊市状态。